In [1]:
# Cell 1: Load TUM freiburg1_desk data + associate RGB/depth/groundtruth by timestamp

import os
import numpy as np

# --- Config ---
DATASET_ROOT = "/mnt/d/My Projects/slam-track-fusion/Data/rgbd_dataset_freiburg1_desk"
MAX_TIME_DIFF = 0.02  # seconds — TUM's own recommended association threshold

RGB_TXT = os.path.join(DATASET_ROOT, "rgb.txt")
DEPTH_TXT = os.path.join(DATASET_ROOT, "depth.txt")
GT_TXT = os.path.join(DATASET_ROOT, "groundtruth.txt")


def read_tum_file(path):
    """
    Parses a TUM-format file. Lines look like:
        <timestamp> <value1> <value2> ...
    Comment lines start with '#' and are skipped.
    Returns a list of (timestamp: float, values: list[str]) tuples,
    sorted by timestamp.
    """
    entries = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            timestamp = float(parts[0])
            values = parts[1:]
            entries.append((timestamp, values))
    entries.sort(key=lambda x: x[0])
    return entries


def associate(reference, other, max_diff=MAX_TIME_DIFF):
    """
    For each entry in `reference`, finds the closest-in-time entry in `other`
    (nearest-neighbor, not interpolation) within max_diff seconds.
    Both inputs are lists of (timestamp, values) as returned by read_tum_file.

    Returns a list of tuples: (ref_timestamp, ref_values, other_timestamp, other_values)
    Only matches within the threshold are kept — unmatched reference entries are dropped.
    """
    other_timestamps = np.array([t for t, _ in other])
    matches = []

    for ref_t, ref_v in reference:
        diffs = np.abs(other_timestamps - ref_t)
        best_idx = np.argmin(diffs)
        best_diff = diffs[best_idx]

        if best_diff <= max_diff:
            other_t, other_v = other[best_idx]
            matches.append((ref_t, ref_v, other_t, other_v))

    return matches


# --- Load all three streams ---
rgb_entries = read_tum_file(RGB_TXT)
depth_entries = read_tum_file(DEPTH_TXT)
gt_entries = read_tum_file(GT_TXT)

print(f"Loaded {len(rgb_entries)} RGB frames")
print(f"Loaded {len(depth_entries)} depth frames")
print(f"Loaded {len(gt_entries)} groundtruth poses")

# --- Associate RGB -> depth, using RGB timestamps as the reference clock ---
rgb_depth_matches = associate(rgb_entries, depth_entries)
print(f"RGB<->depth matches: {len(rgb_depth_matches)} / {len(rgb_entries)} RGB frames")

# --- Associate the RGB-depth pairs -> groundtruth poses ---
# Treat rgb_depth_matches as the new reference stream (using rgb timestamp),
# and associate against groundtruth.
associated = []
gt_timestamps = np.array([t for t, _ in gt_entries])

for rgb_t, rgb_v, depth_t, depth_v in rgb_depth_matches:
    diffs = np.abs(gt_timestamps - rgb_t)
    best_idx = np.argmin(diffs)
    if diffs[best_idx] <= MAX_TIME_DIFF:
        gt_t, gt_v = gt_entries[best_idx]
        associated.append({
            "rgb_timestamp": rgb_t,
            "rgb_file": os.path.join(DATASET_ROOT, rgb_v[0]),
            "depth_timestamp": depth_t,
            "depth_file": os.path.join(DATASET_ROOT, depth_v[0]),
            "gt_timestamp": gt_t,
            "gt_pose": [float(x) for x in gt_v],  # tx ty tz qx qy qz qw
        })

print(f"Fully associated frames (RGB+depth+GT): {len(associated)}")
print("\nFirst associated entry:")
print(associated[0])

Loaded 613 RGB frames
Loaded 595 depth frames
Loaded 2335 groundtruth poses
RGB<->depth matches: 596 / 613 RGB frames
Fully associated frames (RGB+depth+GT): 596

First associated entry:
{'rgb_timestamp': 1305031453.359684, 'rgb_file': '/mnt/d/My Projects/slam-track-fusion/Data/rgbd_dataset_freiburg1_desk/rgb/1305031453.359684.png', 'depth_timestamp': 1305031453.374112, 'depth_file': '/mnt/d/My Projects/slam-track-fusion/Data/rgbd_dataset_freiburg1_desk/depth/1305031453.374112.png', 'gt_timestamp': 1305031453.3595, 'gt_pose': [1.3112, 0.8507, 1.5186, 0.8851, 0.2362, -0.0898, -0.3909]}


In [2]:
# Cell 2: Load pyslam Config + dataset via pyslam's native TUM loader

import sys
import os

# Make pyslam importable regardless of where this notebook lives
PYSLAM_ROOT = os.path.expanduser("~/pyslam")
if PYSLAM_ROOT not in sys.path:
    sys.path.insert(0, PYSLAM_ROOT)

from pyslam.config import Config
from pyslam.io.dataset_factory import dataset_factory
from pyslam.io.ground_truth import groundtruth_factory

# --- Load config.yaml (already edited to point at freiburg1_desk) ---
config = Config(
    config_path=os.path.join(PYSLAM_ROOT, "config.yaml"),
    config_libs_path=os.path.join(PYSLAM_ROOT, "config_libs.yaml"),
)

print("Dataset type:", config.dataset_settings.get("type"))
print("Base path:", config.dataset_settings.get("base_path"))
print("Name:", config.dataset_settings.get("name"))
print("Settings file:", config.dataset_settings.get("settings"))

# --- Build the dataset object (handles RGB/depth loading + association internally) ---
dataset = dataset_factory(config)

# --- Build the groundtruth object ---
groundtruth = groundtruth_factory(config.dataset_settings, cam_settings=config.cam_settings)

print("\nSensor type:", dataset.sensor_type)
print("Number of frames:", dataset.num_frames)
print("Dataset is_ok:", dataset.is_ok)

# --- Sanity check: pull the first frame ---
first_color = dataset.getImageColor(0)
first_depth = dataset.getDepth(0)

print("\nFirst color frame shape:", None if first_color is None else first_color.shape)
print("First depth frame shape:", None if first_depth is None else first_depth.shape)

Dataset type: tum
Base path: /mnt/d/My Projects/slam-track-fusion/Data
Name: rgbd_dataset_freiburg1_desk
Settings file: settings/TUM1.yaml
 dataset_factory - sensor_type: RGBD
Processing TUM Sequence
using groundtruth:  tum
[TumGroundTruth] base_path:  /mnt/d/My Projects/slam-track-fusion/Data/rgbd_dataset_freiburg1_desk

Sensor type: RGBD
Number of frames: 573
Dataset is_ok: True

First color frame shape: (480, 640, 3)
First depth frame shape: (480, 640)


In [3]:
# Cell 3: Initialize pyslam's SLAM system (headless, no GUI)

from pyslam.slam.slam import Slam, SlamState
from pyslam.slam import PinholeCamera
from pyslam.local_features.feature_tracker_configs import FeatureTrackerConfigs
from pyslam.loop_closing.loop_detector_configs import LoopDetectorConfigs
from pyslam.io.ground_truth import is_valid_groundtruth

# --- Camera model from our TUM1.yaml intrinsics (already loaded via config) ---
camera = PinholeCamera(config)
print("Camera:", camera.to_json() if hasattr(camera, "to_json") else camera)

# --- Feature tracker + loop closing configs (using pyslam's recommended defaults) ---
feature_tracker_config = FeatureTrackerConfigs.ORB2
loop_detection_config = LoopDetectorConfigs.DBOW3
semantic_mapping_config = None  # not needed for this project

# --- Apply any overrides from the settings file (mirrors main_slam.py behavior) ---
if config.feature_tracker_config_name is not None:
    feature_tracker_config = FeatureTrackerConfigs.get_config_from_name(
        config.feature_tracker_config_name
    )
if config.num_features_to_extract > 0:
    feature_tracker_config["num_features"] = config.num_features_to_extract
if config.loop_detection_config_name is not None:
    loop_detection_config = LoopDetectorConfigs.get_config_from_name(
        config.loop_detection_config_name
    )

config.feature_tracker_config = feature_tracker_config
config.loop_detection_config = loop_detection_config
config.semantic_mapping_config = semantic_mapping_config

# --- Create the SLAM object (headless=True: no GUI/viewer) ---
slam = Slam(
    camera,
    feature_tracker_config,
    loop_detection_config,
    semantic_mapping_config,
    dataset.sensorType(),
    environment_type=dataset.environmentType(),
    config=config,
    headless=True,
)

has_groundtruth = is_valid_groundtruth(groundtruth)
print("\nSLAM object created.")
print("Has groundtruth:", has_groundtruth)

✅ cpp_module imported successfully, C++ core is available


I0000 00:00:1784021548.676200  113943 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


DepthEstimatorCrestereoMegengine: megengine not found
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Camera: {'D': [0.262383, -0.953104, -0.005358, 0.002628, 1.16331], 'K': [[517.306408, 0.0, 318.64304], [0.0, 516.469215, 255.313989], [0.0, 0.0, 1.0]], 'Kinv': [[0.0019330903010967534, 0.0, -0.6159657701359847], [0.0, 0.0019362238270097087, -0.49434502887069465], [0.0, 0.0, 1.0]], 'b': 0.077324, 'bf': 40.0, 'cx': 318.643, 'cy': 255.314, 'depth_factor': 0.0002, 'depth_threshold': 3.09294, 'fps': 30, 'fx': 517.306, 'fy': 516.469, 'height': 480, 'initialized': True, 'is_distorted': True, 'sensor_type': 'SensorType.RGBD', 'type': 1, 'u_max': 626.048, 'u_min': 10.8012, 'v_max': 473.312, 'v_min': 14.6686, 'width': 640}
FeatureManager: num_levels:  8
FeatureManager: Using opencv  4.10.0
Camera: bf = 40, b = 0.0773236, sensor_type = 2
Using Orbslam2Feature2D
 FeatureManager:

In [4]:
# Cell 4: Run the SLAM tracking loop over all frames, collect poses

import numpy as np

poses = []  # list of dicts: {frame_id, timestamp, R (3x3 list), t (3 list)}
num_tracking_lost = 0

num_total_frames = dataset.num_frames
print(f"Running SLAM over {num_total_frames} frames...")

for img_id in range(num_total_frames):
    if not dataset.is_ok:
        print(f"Dataset stopped being OK at frame {img_id}, stopping early.")
        break

    img = dataset.getImageColor(img_id)
    depth = dataset.getDepth(img_id)

    if img is None:
        print(f"Frame {img_id}: no image, skipping.")
        continue

    timestamp = dataset.getTimestamp()

    # img_right=None since we're RGBD, not stereo
    slam.track(img, None, depth, img_id, timestamp)

    if slam.tracking.state == SlamState.LOST:
        num_tracking_lost += 1

    if slam.tracking.cur_R is not None and slam.tracking.cur_t is not None:
        poses.append({
            "frame_id": img_id,
            "timestamp": timestamp,
            "R": slam.tracking.cur_R.tolist(),
            "t": slam.tracking.cur_t.tolist(),
        })

    if img_id % 50 == 0:
        print(f"Frame {img_id}/{num_total_frames} — poses collected so far: {len(poses)}")

print(f"\nDone. Total poses collected: {len(poses)} / {num_total_frames}")
print(f"Frames where tracking was LOST: {num_tracking_lost}")

Running SLAM over 573 frames...
 @tracking RGBD, img id: 0, frame id: 0, state: NO_IMAGES_YET
img.shape: (480, 640, 3), camera: 480x640
depth.shape:  (480, 640)
timestamp:  1305031453.359684
detector: ORB2 , #features: 1990
descriptor: ORB2 , #features: 1990
detector: ORB2 , descriptor: ORB2 , #features: 1990  (#ref: 2000 ), [kp-filter: NONE ]
Frame 0/573 — poses collected so far: 0
 @tracking RGBD, img id: 1, frame id: 1, state: NOT_INITIALIZED
img.shape: (480, 640, 3), camera: 480x640
depth.shape:  (480, 640)
timestamp:  1305031453.39169
detector: ORB2 , #features: 2000
descriptor: ORB2 , #features: 2000
detector: ORB2 , descriptor: ORB2 , #features: 2000  (#ref: 2000 ), [kp-filter: NONE ]
|------------
Initializer: processing frames: 1, 0
Initializer: # keypoint matches:  907
Initializer: # keypoint inliers:  230
Initializer: pts3d: (230, 3), mask_pts3d: (230,)
Initializer: init PnP - #pts3d: 230, #mask_pts3d: 225
Initializer: init PnP - success: True
Initializer: # triangulated poi

local_bundle_adjustment: starting optimization with 32 keyframes, 2028 points, 5319 edges
 @tracking RGBD, img id: 162, frame id: 162, state: OK
img.shape: (480, 640, 3), camera: 480x640
depth.shape:  (480, 640)
timestamp:  1305031459.25976
detector: ORB2 , #features: 1000
descriptor: ORB2 , #features: 1000
detector: ORB2 , descriptor: ORB2 , #features: 1000  (#ref: 1000 ), [kp-filter: NONE ]
using motion model for next pose prediction
>>>> tracking previous frame ...
search frame by projection
# matched map points in prev frame: 381 
pose opt proj-frame-frame 
pose_optimization: available 381 points, found 369 bad points
     error^2: 2.898408,  ok: 1
descriptor sigma:  130.0
>>>> tracking local map...
# matched map points in local map: 58, perc%: 2.57
pose opt proj-map-frame 
     error^2: 2.951108,  ok: 1
pose_optimization: available 70 points, found 32 bad points
 F(162) #matched points: 38, KF(161) #matched points: 329
is_local_mapping_idle:  False , local_mapping_queue_size:  0
K

local_bundle_adjustment: starting optimization with 34 keyframes, 2788 points, 9535 edges
 @tracking RGBD, img id: 166, frame id: 166, state: OK
img.shape: (480, 640, 3), camera: 480x640
depth.shape:  (480, 640)
timestamp:  1305031459.427646
detector: ORB2 , #features: 1000
descriptor: ORB2 , #features: 1000
detector: ORB2 , descriptor: ORB2 , #features: 1000  (#ref: 1000 ), [kp-filter: NONE ]
using motion model for next pose prediction
>>>> tracking previous frame ...
search frame by projection
# matched map points in prev frame: 492 
pose opt proj-frame-frame 
pose_optimization: available 490 points, found 409 bad points
     error^2: 3.560659,  ok: 1
descriptor sigma:  130.0
>>>> tracking local map...
# matched map points in local map: 96, perc%: 3.45
pose opt proj-map-frame 
     error^2: 3.229065,  ok: 1
pose_optimization: available 177 points, found 55 bad points
 F(166) #matched points: 122, KF(165) #matched points: 210
is_local_mapping_idle:  False , local_mapping_queue_size:  

# keypoints matched: 80 
descriptor sigma:  129.23145
# matched map points in reference frame: 75 
pose opt match-frame-frame 
pose_optimization: available 93 points, found 51 bad points
     error^2: 2.767129,  ok: 1
>>>> tracking local map...
# matched map points in local map: 164, perc%: 5.18
pose opt proj-map-frame 
pose_optimization: available 206 points, found 91 bad points
     error^2: 3.135833,  ok: 1
 F(168) #matched points: 115, KF(167) #matched points: 225
is_local_mapping_idle:  False , local_mapping_queue_size:  0
KF conditions: ( (1a:False or 1b:False or 1c:True or 1d:False) and 2: True ) or 3: False
 interrupting local mapping optimization
 map: 4527 points, 35 keyframes
Tracking: elapsed_time:  0.039669036865234375
 KF(167) #points: 207 
 @tracking RGBD, img id: 169, frame id: 169, state: OK
img.shape: (480, 640, 3), camera: 480x640
depth.shape:  (480, 640)
timestamp:  1305031459.559647
detector: ORB2 , #features: 1000
descriptor: ORB2 , #features: 1000
detector: ORB2 

In [5]:
# Cell 5: Quick sanity check on the collected poses before saving

print(f"Total poses: {len(poses)}")
print("\nFirst pose:")
print(poses[0])
print("\nLast pose:")
print(poses[-1])

# Extract just translations to eyeball the trajectory's spatial extent
translations = np.array([p["t"] for p in poses]).reshape(len(poses), 3)
print("\nTranslation stats (x, y, z):")
print("min:", translations.min(axis=0))
print("max:", translations.max(axis=0))
print("range:", translations.max(axis=0) - translations.min(axis=0))

Total poses: 571

First pose:
{'frame_id': 2, 'timestamp': 1305031453.423683, 'R': [[0.9987830698778122, -0.04324619143729892, 0.02370962360776616], [0.04325859191972517, 0.9990639089280079, -1.0129750762369935e-05], [-0.023686991167645954, 0.0010357623557821754, 0.999718887310711]], 't': [0.0025945574204092785, 0.006075014809248019, -0.01726871545237979]}

Last pose:
{'frame_id': 572, 'timestamp': 1305031473.196069, 'R': [[0.9989159304444928, 0.04214718308455439, -0.01976306813858245], [-0.038800328218235, 0.9884106371996648, 0.1467615303841912], [0.02571961186274857, -0.14583561714680573, 0.988974455856692]], 't': [-0.8445225577355332, 0.4711146718451617, -0.2219787879889955]}

Translation stats (x, y, z):
min: [-2.22665133 -0.45886587 -0.74903558]
max: [0.01465884 0.71189057 0.56422287]
range: [2.24131017 1.17075644 1.31325845]


In [7]:
# Cell 6: Save SLAM outputs — poses.json, trajectory.txt (TUM format), map_points.npy

import json

OUTPUT_DIR = "/mnt/d/My Projects/slam-track-fusion/Output"  # changed from DATASET_ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. poses.json — primary, JSON-serializable, for FastAPI use later ---
poses_json_path = os.path.join(OUTPUT_DIR, "poses.json")
with open(poses_json_path, "w") as f:
    json.dump(poses, f, indent=2)
print(f"Saved poses.json ({len(poses)} poses) to {poses_json_path}")


# --- 2. trajectory.txt — TUM format: timestamp tx ty tz qx qy qz qw ---
def rotation_matrix_to_quaternion(R):
    """Converts a 3x3 rotation matrix to a quaternion [qx, qy, qz, qw]."""
    R = np.array(R)
    trace = np.trace(R)
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        qw = 0.25 / s
        qx = (R[2, 1] - R[1, 2]) * s
        qy = (R[0, 2] - R[2, 0]) * s
        qz = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        qw = (R[2, 1] - R[1, 2]) / s
        qx = 0.25 * s
        qy = (R[0, 1] + R[1, 0]) / s
        qz = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        qw = (R[0, 2] - R[2, 0]) / s
        qx = (R[0, 1] + R[1, 0]) / s
        qy = 0.25 * s
        qz = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        qw = (R[1, 0] - R[0, 1]) / s
        qx = (R[0, 2] + R[2, 0]) / s
        qy = (R[1, 2] + R[2, 1]) / s
        qz = 0.25 * s
    return [qx, qy, qz, qw]


trajectory_txt_path = os.path.join(OUTPUT_DIR, "trajectory.txt")
with open(trajectory_txt_path, "w") as f:
    f.write("# timestamp tx ty tz qx qy qz qw\n")
    for p in poses:
        qx, qy, qz, qw = rotation_matrix_to_quaternion(p["R"])
        tx, ty, tz = p["t"]
        f.write(f"{p['timestamp']:.6f} {tx:.6f} {ty:.6f} {tz:.6f} {qx:.6f} {qy:.6f} {qz:.6f} {qw:.6f}\n")
print(f"Saved trajectory.txt ({len(poses)} lines) to {trajectory_txt_path}")


# --- 3. map_points.npy — sparse 3D map from SLAM ---
map_points_path = os.path.join(OUTPUT_DIR, "map_points.npy")
map_points_3d = np.array([mp.pt() for mp in slam.map.get_points() if mp is not None])
np.save(map_points_path, map_points_3d)
print(f"Saved map_points.npy ({map_points_3d.shape[0]} points, shape {map_points_3d.shape}) to {map_points_path}")

Saved poses.json (571 poses) to /mnt/d/My Projects/slam-track-fusion/Output/poses.json
Saved trajectory.txt (571 lines) to /mnt/d/My Projects/slam-track-fusion/Output/trajectory.txt
Saved map_points.npy (7808 points, shape (7808, 3)) to /mnt/d/My Projects/slam-track-fusion/Output/map_points.npy
